# plant-borb on Google Colab

BorB segmentation + CNN plant-disease classification (Exp 1 / Exp 2 / Exp 3).

Runtime → Change runtime type → **GPU** (T4 is enough; batch 16 + AMP fits in < 6 GB).

In [ ]:
# 1) Get the code: upload the plant-borb folder to Drive, or clone your repo
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/plant-borb

In [ ]:
# 2) Install deps (Colab already has torch/torchvision with CUDA)
!pip -q install 'timm>=1.0.9' 'albumentations>=2.0,<3' imagehash thop tabulate pyarrow

In [ ]:
# 3) Smoke test (synthetic data, ~1 min)
!python run.py --smoke
!python -m pytest -q

In [ ]:
# 4) Data: downloaded from public GitHub/HuggingFace mirrors (no account or API key needed).
!python scripts/download_data.py

In [ ]:
# 5) Splits + segmentation QC (thresholds from the TRAIN split only)
!python scripts/segmentation_report.py --config configs/exp1_borb.yaml

In [ ]:
# 6) Exp 1 / Exp 2 with one backbone first (num_workers=2 is safest on Colab)
!python run.py --config configs/exp1_borb.yaml --backbones efficientnet_b3 --override num_workers=2
!python run.py --config configs/exp2_noborb.yaml --backbones efficientnet_b3 --override num_workers=2

In [ ]:
# 7) Robustness + summary tables
!python run.py --config configs/exp3_robustness.yaml --backbones efficientnet_b3 --override num_workers=2
from IPython.display import Markdown; Markdown(open('results/summary.md').read())

In [ ]:
# 8) Full grid (long! ~4 backbones x 3 seeds x 2 experiments)
# !python run.py --config configs/exp1_borb.yaml configs/exp2_noborb.yaml --backbones vgg16,resnet50,efficientnet_b3,mobilenetv3_large,efficientnetv2_s,convnext_tiny
# !python run.py --config configs/exp3_robustness.yaml --backbones vgg16,resnet50,efficientnet_b3,mobilenetv3_large,efficientnetv2_s,convnext_tiny